# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step exploration of the FAIR^2 colorectal cancer dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the FAIR^2 dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pprint

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset name:", metadata.name)
print("Dataset description:", metadata.description)
print("Identifier:", metadata.identifier)
print("Date published:", metadata.datePublished)
print("Record sets in metadata:", metadata.recordSet)


## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant schema.


In [ ]:
# List all available record sets from the metadata
# Each record set, field, and column is referenced by its @id

print("Available record sets (by @id):")
record_sets = []
for rs in dataset.record_sets():
    print(f"  RecordSet @id: {rs['@id']}  Name: {rs.get('name', '<no name>')}")
    record_sets.append(rs['@id'])
    if 'field' in rs:
        print("    Fields:")
        for field in rs['field']:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f"      Field @id: {field_id}")
            # Optionally show columns if available
            if isinstance(field, dict) and 'column' in field:
                for col in field['column']:
                    col_id = col['@id'] if isinstance(col, dict) and '@id' in col else col
                    print(f"        Column @id: {col_id}")
print("\nFull list of record_set @id's:", record_sets)


## 3. Data Extraction
Load table data from each record set into DataFrames for analysis. All access to record sets and fields uses their `@id`.


In [ ]:
# Extract records from each record set
# We use only the first available record set (for simplicity) if there is only one
dataframes = {}
for rs_id in record_sets:
    print(f"Loading records for RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns in RecordSet {rs_id}:", df.columns.tolist())
    print(df.head(3), "\n")

# For subsequent steps, pick the first record set
selected_rs_id = record_sets[0] if record_sets else None

# Display columns for reference
if selected_rs_id and selected_rs_id in dataframes:
    print("Columns in selected RecordSet:", dataframes[selected_rs_id].columns.tolist())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping by key attributes.
All columns are referenced using their field or column `@id`.

In [ ]:
# Choose numeric and group fields by inspecting column names

df = dataframes[selected_rs_id]
numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != 'id']

print("Numeric fields:", numeric_fields)
print("Group fields:", group_fields)

# For demonstration, choose the first numeric and group field
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = None

if group_fields:
    group_field_id = group_fields[0]
else:
    group_field_id = None

# Example filtering for values greater than threshold
if numeric_field_id:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships using `matplotlib` and `seaborn`.

In [ ]:
# Plotting histograms and group distributions
if selected_rs_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Grouped boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- This notebook used `mlcroissant` to load and examine the FAIR^2 colorectal cancer dataset.
- We reviewed available record sets and fields using their `@id` references.
- Extracted and analyzed tabular data, applied basic filtering and normalization, and visualized variable distributions.
- For more robust analysis, explore additional record sets and fields, and consult the schema for detailed data lineage and provenance.
